In [13]:
import pandas as pd
from internalization_backtest import BacktestConfig, HoldToEOD, TimeBucketNetting, ToleranceBandPartialHedge, ProbabilisticWait, run_backtest

# flow_df 必含列：ts, instrument, side, qty, px, mid, client_id（ts需为带时区的Timestamp）
cfg = BacktestConfig(instruments=["EURUSD"], half_spread_bps=0.5, impact_bps_per_mm=0.1, inv_penalty_per_mm=0.5)

res_eod  = run_backtest(flow_df, HoldToEOD(pd.Timestamp("2025-09-24 18:00", tz="Europe/London")), cfg)
res_tb   = run_backtest(flow_df, TimeBucketNetting(bucket_minutes=30, tolerance_notional=3e5), cfg)
res_band = run_backtest(flow_df, ToleranceBandPartialHedge(band_notional=5e5, decay=0.6), cfg)
res_prob = run_backtest(flow_df, ProbabilisticWait(horizon_minutes=15, p_hold_threshold=0.65, hedge_fraction=0.6), cfg)


NameError: name 'flow_df' is not defined

In [16]:
import pandas as pd
from internalization_backtest import BacktestConfig, run_backtest
from combined_strategy import CombinedToleranceProb

# flow_df: 需要列 ts, instrument, side, qty, px, mid, client_id（ts建议带 Europe/London 时区）
cfg = BacktestConfig(instruments=["EURUSD"], half_spread_bps=0.5, impact_bps_per_mm=0.1, inv_penalty_per_mm=0.5)

strategy = CombinedToleranceProb(
    band_notional=1_000_000,                # 容忍带（名义）
    layers=[(1.0, 0.4), (2.0, 0.7), (3.0, 1.0)],  # 分层：超带1x/2x/3x时，对超出部分对冲比例
    horizon_minutes=15,                     # 预测窗口
    hold_threshold=0.65,                    # 若 P(对手盘到来) >= 阈值 → 倾向等待
    wait_discount=0.5,                      # 倾向等待时，将分层对冲比例乘以 (1 - 0.5)
    inside_fraction=0.0,                    # 在带内时，如 P 低于阈值，也可小幅对冲（这里关掉）
    min_trade_qty=50_000,                   # 小于5万的对冲不下，避免碎单
    eod_time=pd.Timestamp("2025-09-24 18:00", tz="Europe/London"),  # 到点全平（可选）
    fallback_lambda_per_min=0.05            # 若没提供λ，就用此常数（每分钟到达率）
)

res = run_backtest(flow_df, strategy, cfg)


NameError: name 'flow_df' is not defined

In [19]:
# use lambda calibrator

from lambda_calibrator import LambdaManager, FlowAtom
from combined_strategy import CombinedToleranceProb
from internalization_backtest import run_backtest, BacktestConfig

lm = LambdaManager()  # Rolling Poisson(15m) + EWMA(10m半衰)

class StrategyWithLambda(CombinedToleranceProb):
    def on_event(self, event, state):
        # 1) 更新到达率校准器
        lm.update(FlowAtom(ts=event.ts, instrument=event.instrument, side=event.side, qty=event.qty))
        # 2) 把“抵消库存”的对手盘λ写进 state，让 CombinedToleranceProb 用它来算 p_arrival
        inv = state["inventory"][event.instrument]
        inv_sign = 1 if inv > 0 else (-1 if inv < 0 else 0)
        if inv_sign != 0:
            lm.write_to_state_for_opposite(state, event.instrument, inv_sign, now=event.ts)
        # 3) 照常走父类逻辑
        return super().on_event(event, state)

# 用法
strategy = StrategyWithLambda(
    band_notional=1_000_000,
    layers=[(1.0, 0.4), (2.0, 0.7), (3.0, 1.0)],
    horizon_minutes=15,
    hold_threshold=0.65,
    wait_discount=0.5,
    inside_fraction=0.0,
    min_trade_qty=50_000,
    eod_time=pd.Timestamp("2025-09-24 18:00", tz="Europe/London"),
    fallback_lambda_per_min=0.02  # 只有在还没估出来时才会用到
)

# 然后像原来一样 run_backtest(flow_df, strategy, cfg)


In [ ]:
import pandas as pd
from internalization_backtest import BacktestConfig, run_backtest
from strategy_tp_sl import HybridInternalTPSSL, TPSSLParams

cfg = BacktestConfig(instruments=["EURUSD"], half_spread_bps=0.5, impact_bps_per_mm=0.1, inv_penalty_per_mm=0.5)

params = TPSSLParams(
    tp_bps=5.0,                 # 触发止盈：相对入场锚点+5 bps
    sl_bps=8.0,                 # 触发止损：相对入场锚点-8 bps（多头），空头相反
    tp_fraction=0.5,            # 命中TP时，对当前仓位对冲50%
    sl_fraction=0.7,            # 命中SL时，对当前仓位对冲70%
    trailing_tp=True,           # 可选：止盈端跟踪（价格更有利时抬高/压低锚点）
    trailing_sl=False,          # 可选：是否做跟踪止损
    min_trade_qty=50_000,       # 小于5万的对冲不下单
    tolerance_band_notional=3e5,# 带内（30万名义）先不对冲，偏向等待 internalize
    cooldown_minutes=2.0        # 两分钟内不重复对冲（去噪）
)

strategy = HybridInternalTPSSL(params, eod_time=pd.Timestamp("2025-09-24 18:00", tz="Europe/London"))

# 跑
res = run_backtest(flow_df, strategy, cfg)


In [ ]:
# use strategy TP_SL

# tp_sl_with_lambda.py (粘贴到你的工程里即可)
from strategy_tp_sl import HybridInternalTPSSL, TPSSLParams
from lambda_calibrator import LambdaManager, FlowAtom
import numpy as np

class HybridTPSSLWithLambda(HybridInternalTPSSL):
    def __init__(self, params: TPSSLParams, 
                 eod_time=None,
                 horizon_minutes: int = 15,
                 hold_threshold: float = 0.65,   # p_arrival >= 阈值 → 倾向“等”
                 inside_fraction: float = 0.2,   # p_arrival < 阈值时，带内预对冲比例
                 fallback_lambda_per_min: float = 0.02):
        super().__init__(params, eod_time=eod_time)
        self.horizon = horizon_minutes
        self.hold_threshold = hold_threshold
        self.inside_fraction = inside_fraction
        self.fallback_lambda = fallback_lambda_per_min
        self.lm = LambdaManager()  # RollingPoisson + EWMA

    def _p_arrival(self, instrument: str, state, now):
        lam = state.get("lambda_per_minute", {}).get(instrument, self.fallback_lambda)
        return 1 - np.exp(-lam * self.horizon)

    def on_event(self, event, state):
        inst = event.instrument
        mid  = event.mid

        # 1) 先更新 λ 校准器，并把“抵消库存”的λ写回 state
        self.lm.update(FlowAtom(ts=event.ts, instrument=inst, side=event.side, qty=event.qty))
        inv = state["inventory"][inst]
        inv_sign = (1 if inv > 0 else (-1 if inv < 0 else 0))
        if inv_sign != 0:
            self.lm.write_to_state_for_opposite(state, inst, inv_sign, now=event.ts)

        # 2) 先走父类的内部逻辑（包含容忍带 + TP/SL）
        actions = super().on_event(event, state)
        if actions: 
            return actions

        # 3) 仅当“容忍带”拦住了对冲（即当前还在带内时），根据 p_arrival 做预对冲
        if self.p.tolerance_band_notional > 0 and inv != 0:
            notional = abs(inv) * mid
            if notional <= self.p.tolerance_band_notional and self.inside_fraction > 0:
                p_arr = self._p_arrival(inst, state, event.ts)
                if p_arr < self.hold_threshold and self._cooldown_ok(inst, event.ts):
                    qty = inv * self.inside_fraction
                    if abs(qty) >= self.p.min_trade_qty:
                        self.last_hedge_ts[inst] = event.ts
                        return [(inst, qty)]

        return []


In [2]:
import pandas as pd
from internalization_backtest import BacktestConfig
from strategy_tp_sl_v2 import HybridInternalTPSSL, TPSSLParams
from run_with_quotes_support import run_backtest_with_quotes

# 读取Mock数据
price_df = pd.read_csv("price_path.csv", parse_dates=["ts"])
trades_df = pd.read_csv("client_trades.csv", parse_dates=["ts"])

cfg = BacktestConfig(instruments=["EURUSD"], half_spread_bps=0.5, impact_bps_per_mm=0.1, inv_penalty_per_mm=0.5)

params = TPSSLParams(
    tp_bps=5, sl_bps=8, tp_fraction=0.5, sl_fraction=0.7,
    trailing_tp=True, trailing_sl=False,
    min_trade_qty=50_000, tolerance_band_notional=3e5, cooldown_minutes=2
)

strategy = HybridInternalTPSSL(params, eod_time=pd.Timestamp("2025-09-24 18:00", tz="Europe/London"))

res = run_backtest_with_quotes(trades_df, price_df, strategy, cfg)
print(res["metrics"])


ModuleNotFoundError: No module named 'internalization_backtest'

In [4]:
import os
os.getcwd()

'/Users/qinli/Documents/GitHub/python_tricks'